The Scenario: You are given three separate datasets: "Customer Profiles" (age, region, subscription tier), "Product Catalog" (category, weight, unit price), and "Order Transactions" (who bought what, when, and delivery times).

In [ ]:
import pandas as pd

df1 = pd.read_csv("customers - customers.csv")
df2 = pd.read_csv("orders_feb - orders_feb.csv")
df3 = pd.read_csv("orders_jan - orders_jan.csv")
df4 = pd.read_csv("products - products.csv")
[df1, df2, df3, df4]

orders = pd.concat([jan, feb], ignore_index=True)
print(orders.head())



orders = pd.merge(
    orders,
    products,
    on="product_id",
    how="left"
)

print(orders.head())


orders = pd.merge(
    orders,
    customers,
    on="customer_id",
    how="left"
)

print(orders.head())



orders = orders.rename(columns={
    "cust_ID": "Customer_ID",
    "unit_price": "Unit_Price",
    "quantity": "Quantity",
    "delivery_days": "Delivery_Days"
})


orders = orders.drop(columns=["warehouse_routing_id"], errors="ignore")


orders["Total_Revenue"] = orders["Quantity"] * orders["Unit_Price"]

print(orders.head())


# =========================
# 6. APPLY FUNCTION (DELIVERY PERFORMANCE)
# =========================

def delivery_status(days):
    if days <= 2:
        return "Early"
    elif days <= 4:
        return "On-Time"
    else:
        return "Delayed"

orders["Delivery_Performance"] = orders["Delivery_Days"].apply(delivery_status)

print(orders[["Delivery_Days", "Delivery_Performance"]].head())


# =========================
# 7. GROUPBY + AGGREGATION
# =========================
category_summary = orders.groupby("Product_Category").agg(
    total_revenue=("Total_Revenue", "sum"),
    avg_delivery_days=("Delivery_Days", "mean"),
    unique_customers=("Customer_ID", "nunique")
)

print(category_summary)


# =========================
# 8. PIVOT TABLE (BUSINESS MATRIX)
# =========================
pivot_report = pd.pivot_table(
    orders,
    index="Customer Region",
    columns="Delivery_Performance",
    values="Total_Revenue",
    aggfunc="sum",
    fill_value=0
)

print(pivot_report)

Combining Datasets: You must use pd.concat() to stitch together the order transactions from "January" and "February". Then, use pd.merge() to join the combined 'Order Transactions' table with the 'Product Catalog' table so you know the category and price of the item purchased, acting like a SQL left join.

Modifying DataFrames: You create a new calculated column called Total_Revenue by multiplying the Quantity column by the Unit_Price column. Use .drop() to remove redundant warehouse routing IDs, and use .rename() to fix inconsistently capitalized columns (e.g., changing cust_ID to Customer_ID).

Applying Functions: You write a custom function (or use a lambda) and apply it via .apply() to categorize the Delivery_Days column into delivery performance buckets: "Early" (1-2 days), "On-Time" (3-4 days), or "Delayed" (5+ days).

Grouping and Aggregation: Using .groupby(), you group the data by Product_Category and use .agg() to find the total revenue generated, the average delivery days, and the unique count of Customer_IDs who purchased from that category.

Reshaping and Pivoting: Finally, you use pd.pivot_table() to create a business-reporting matrix showing "Customer Region" as rows, "Delivery Performance" (Early/On-Time/Delayed) as columns, and the "Total Revenue" as the values.